In [1]:
# Task 1: List 8 business questions Olist's leadership would ask
from pathlib import Path

Path("README.md").write_text(r'''# Day 16 - Olist warehouse model

I kept this simple and human. The answers to the "thinking" questions are in Markdown files; the database work is in `.sql` files.

Files:
- `business_questions.md` - the 8 questions leadership would ask
- `model.md` - grain, measures, dimensions and keys
- `star_schema.md` - the star diagram in Mermaid (draw.io replacement)
- `dim_date.sql` - recursive CTE for the date dimension
- `build_tables.sql` - staging tables, dimensions and `fact_sales`
- `check_totals.sql` - SQL to compare fact totals with source totals
- `fact_reviews.sql` and `fact_reviews.md` - reviews as a separate fact

Note: MySQL Workbench and draw.io are desktop GUI tools and are out of scope here. I wrote the diagram as Mermaid and the database work as plain SQL instead. The `mysql` command line is also not installed locally, so the SQL is ready to run but was not executed in this environment.
''', encoding="utf-8")

Path("business_questions.md").write_text(r'''# What Olist leadership would probably ask

These are written the way a busy manager would actually ask them, not like exam questions.

1. How much money did we really make each month, and are we growing?
2. Which product categories sell best, and which ones just sit there?
3. Which states and customers bring in the most money, so where should we spend marketing budget?
4. How long does an order take from purchase to delivery, and where does it get stuck?
5. Which sellers are quick and reliable, and which ones need a conversation?
6. How much of our revenue disappears into freight, and is it hurting our margin?
7. How happy are customers, and what do they complain about most often?
8. How do people pay, and do installments lead to bigger orders?

Every question above can be answered from the star schema below.
''', encoding="utf-8")

print("Wrote README.md and business_questions.md")

Wrote README.md and business_questions.md


In [2]:
# Task 2: Define the grain of fact_sales (one row per order item)
from pathlib import Path

Path("model.md").write_text(r'''# The Olist sales model, in plain words

## Grain of fact_sales

One row = one item on one order. If a customer buys three items, `fact_sales` gets three rows.

I did not make it one row per order because price, freight and seller are different for every item. Rolling up to the order would hide exactly the detail leadership asks for: which product sold, which seller shipped it, and how much freight it cost.
''', encoding="utf-8")

print("Wrote model.md with the grain")

Wrote model.md with the grain


In [3]:
# Task 3: List the measures
with open("model.md", "a", encoding="utf-8") as file:
    file.write(r'''
## Measures (the numbers we add up)

- `price` - what the item sold for
- `freight_value` - shipping cost for that item
- `item_value` - simply price + freight, handy for quick reports
- `item_count` - always 1 per row, so we can count items with SUM instead of COUNT

I deliberately left `payment_value` out of `fact_sales`. Payments are stored per order, not per item, so putting that number on each item row would double count it. It belongs in its own payment fact if we need it later.
''')

print("Added measures to model.md")

Added measures to model.md


In [4]:
# Task 4: List the dimensions: customer, product, seller, date, payment type
with open("model.md", "a", encoding="utf-8") as file:
    file.write(r'''
## Dimensions (the "by" columns)

- **customer** - who bought: customer id, unique id, zip prefix, city, state
- **product** - what was bought: product id, category (English name if we have it), size and weight
- **seller** - who sold it: seller id, zip prefix, city, state
- **date** - when it happened: based on `order_purchase_timestamp`, with month, quarter, year and weekday ready for reports
- **payment type** - how they paid: credit card, boleto, voucher or debit card

Each dimension has one row per thing, and `fact_sales` keeps only a small key that points back to it.
''')

print("Added dimensions to model.md")

Added dimensions to model.md


In [5]:
# Task 5: Draw the star schema
from pathlib import Path

Path("star_schema.md").write_text(r'''# Star schema

draw.io is a desktop app, so I drew this as a Mermaid diagram. It shows the same thing: one fact table in the middle with five dimensions around it.

```mermaid
erDiagram
    dim_customer ||--o{ fact_sales : "customer_key"
    dim_product  ||--o{ fact_sales : "product_key"
    dim_seller   ||--o{ fact_sales : "seller_key"
    dim_date     ||--o{ fact_sales : "date_key"
    dim_payment_type ||--o{ fact_sales : "payment_type_key"
    dim_date     ||--o{ fact_reviews : "date_key"
    dim_product  ||--o{ fact_reviews : "product_key"
```

The `||--o{` symbol means "one dimension row connects to many fact rows". For example, one customer can have many sales rows.

`fact_reviews` sits next to the star because it answers different questions (see `fact_reviews.md`).
''', encoding="utf-8")

print("Wrote star_schema.md with a Mermaid diagram")

Wrote star_schema.md with a Mermaid diagram


In [6]:
# Task 6: Decide on surrogate vs natural keys and note why
with open("model.md", "a", encoding="utf-8") as file:
    file.write(r'''
## Surrogate keys vs natural keys

Olist gives us natural keys such as `order_id` and `product_id`, and they are 32-character hex strings. I keep those in the dimensions so we can always trace a row back to the source CSV.

In `fact_sales` I use small integer **surrogate** keys instead. Two reasons: storing a 32-character string millions of times is wasteful, and if Olist ever changes its ID format the surrogate key does not change, so the warehouse stays stable.

For `dim_date` I use `YYYYMMDD` as the key. It is technically a smart surrogate key, but it is short, sortable and easy for a human to read in a report. `dim_payment_type` gets a tiny integer key because the natural value is a long word like `credit_card`.
''')

print("Added key decision to model.md")

Added key decision to model.md


In [7]:
# Task 7: Generate dim_date in MySQL with a recursive CTE
from pathlib import Path

Path("dim_date.sql").write_text(r'''-- Run this in MySQL 8+ before build_tables.sql.
-- The recursive CTE makes one row per day; we then pull out month, quarter, year and weekday.

CREATE TABLE IF NOT EXISTS dim_date (
    date_key INT PRIMARY KEY,
    full_date DATE NOT NULL,
    month TINYINT NOT NULL,
    quarter TINYINT NOT NULL,
    year SMALLINT NOT NULL,
    weekday VARCHAR(10) NOT NULL
);

WITH RECURSIVE days AS (
    SELECT DATE('2016-01-01') AS full_date
    UNION ALL
    SELECT DATE_ADD(full_date, INTERVAL 1 DAY)
    FROM days
    WHERE full_date < '2018-12-31'
)
INSERT INTO dim_date (date_key, full_date, month, quarter, year, weekday)
SELECT
    DATE_FORMAT(full_date, '%Y%m%d'),
    full_date,
    MONTH(full_date),
    QUARTER(full_date),
    YEAR(full_date),
    DAYNAME(full_date)
FROM days;

SELECT COUNT(*) AS days_created FROM dim_date;
''', encoding="utf-8")

print("Wrote dim_date.sql")

Wrote dim_date.sql


In [8]:
# Task 8: Build the fact and dimension tables in MySQL from Olist
from pathlib import Path

Path("build_tables.sql").write_text(r'''-- Run dim_date.sql first. Then run this script in MySQL Workbench.
-- Change the folder in each LOAD DATA line to where you keep the Olist CSVs.
-- LOAD DATA LOCAL INFILE needs the local_infile option enabled on the server/client.

DROP TABLE IF EXISTS fact_sales;
DROP TABLE IF EXISTS dim_customer;
DROP TABLE IF EXISTS dim_product;
DROP TABLE IF EXISTS dim_seller;
DROP TABLE IF EXISTS dim_payment_type;

CREATE TABLE dim_customer (
    customer_key INT AUTO_INCREMENT PRIMARY KEY,
    customer_id CHAR(32),
    customer_unique_id CHAR(32),
    zip_prefix VARCHAR(5),
    city VARCHAR(100),
    state VARCHAR(2)
);

CREATE TABLE dim_product (
    product_key INT AUTO_INCREMENT PRIMARY KEY,
    product_id CHAR(32),
    category_name VARCHAR(100),
    category_name_english VARCHAR(100),
    weight_g INT,
    length_cm INT,
    height_cm INT,
    width_cm INT
);

CREATE TABLE dim_seller (
    seller_key INT AUTO_INCREMENT PRIMARY KEY,
    seller_id CHAR(32),
    zip_prefix VARCHAR(5),
    city VARCHAR(100),
    state VARCHAR(2)
);

CREATE TABLE dim_payment_type (
    payment_type_key INT AUTO_INCREMENT PRIMARY KEY,
    payment_type VARCHAR(30)
);

CREATE TABLE fact_sales (
    sales_key BIGINT AUTO_INCREMENT PRIMARY KEY,
    order_id CHAR(32),
    order_item_id INT,
    customer_key INT,
    product_key INT,
    seller_key INT,
    date_key INT,
    payment_type_key INT,
    price DECIMAL(10,2),
    freight_value DECIMAL(10,2),
    item_count TINYINT DEFAULT 1,
    order_status VARCHAR(30),
    UNIQUE KEY uq_order_item (order_id, order_item_id)
);

-- Staging tables: plain copies of the CSV files.
DROP TABLE IF EXISTS stg_order_items;
CREATE TABLE stg_order_items (
    order_id CHAR(32), order_item_id INT, product_id CHAR(32), seller_id CHAR(32),
    shipping_limit_date DATETIME, price DECIMAL(10,2), freight_value DECIMAL(10,2)
);
LOAD DATA LOCAL INFILE 'C:/path/archive/olist_order_items_dataset.csv'
INTO TABLE stg_order_items FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n' IGNORE 1 LINES;

DROP TABLE IF EXISTS stg_orders;
CREATE TABLE stg_orders (
    order_id CHAR(32), customer_id CHAR(32), order_status VARCHAR(30),
    order_purchase_timestamp DATETIME, order_approved_at DATETIME,
    order_delivered_carrier_date DATETIME, order_delivered_customer_date DATETIME,
    order_estimated_delivery_date DATETIME
);
LOAD DATA LOCAL INFILE 'C:/path/archive/olist_orders_dataset.csv'
INTO TABLE stg_orders FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n' IGNORE 1 LINES;

DROP TABLE IF EXISTS stg_customers;
CREATE TABLE stg_customers (
    customer_id CHAR(32), customer_unique_id CHAR(32),
    customer_zip_code_prefix VARCHAR(5), customer_city VARCHAR(100), customer_state VARCHAR(2)
);
LOAD DATA LOCAL INFILE 'C:/path/archive/olist_customers_dataset.csv'
INTO TABLE stg_customers FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n' IGNORE 1 LINES;

DROP TABLE IF EXISTS stg_sellers;
CREATE TABLE stg_sellers (
    seller_id CHAR(32), seller_zip_code_prefix VARCHAR(5),
    seller_city VARCHAR(100), seller_state VARCHAR(2)
);
LOAD DATA LOCAL INFILE 'C:/path/archive/olist_sellers_dataset.csv'
INTO TABLE stg_sellers FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n' IGNORE 1 LINES;

DROP TABLE IF EXISTS stg_products;
CREATE TABLE stg_products (
    product_id CHAR(32), product_category_name VARCHAR(100),
    product_name_lenght INT, product_description_lenght INT, product_photos_qty INT,
    product_weight_g INT, product_length_cm INT, product_height_cm INT, product_width_cm INT
);
LOAD DATA LOCAL INFILE 'C:/path/archive/olist_products_dataset.csv'
INTO TABLE stg_products FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n' IGNORE 1 LINES;

DROP TABLE IF EXISTS stg_category_translation;
CREATE TABLE stg_category_translation (
    product_category_name VARCHAR(100), product_category_name_english VARCHAR(100)
);
-- This file has a BOM on the first column; strip it if the first category looks wrong.
LOAD DATA LOCAL INFILE 'C:/path/archive/product_category_name_translation.csv'
INTO TABLE stg_category_translation FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n' IGNORE 1 LINES;

DROP TABLE IF EXISTS stg_order_payments;
CREATE TABLE stg_order_payments (
    order_id CHAR(32), payment_sequential INT, payment_type VARCHAR(30),
    payment_installments INT, payment_value DECIMAL(10,2)
);
LOAD DATA LOCAL INFILE 'C:/path/archive/olist_order_payments_dataset.csv'
INTO TABLE stg_order_payments FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n' IGNORE 1 LINES;

-- Fill the dimensions.
INSERT INTO dim_customer (customer_id, customer_unique_id, zip_prefix, city, state)
SELECT DISTINCT customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state
FROM stg_customers;

INSERT INTO dim_seller (seller_id, zip_prefix, city, state)
SELECT DISTINCT seller_id, seller_zip_code_prefix, seller_city, seller_state
FROM stg_sellers;

INSERT INTO dim_product (product_id, category_name, category_name_english, weight_g, length_cm, height_cm, width_cm)
SELECT p.product_id, p.product_category_name, t.product_category_name_english,
       p.product_weight_g, p.product_length_cm, p.product_height_cm, p.product_width_cm
FROM stg_products p
LEFT JOIN stg_category_translation t ON t.product_category_name = p.product_category_name;

INSERT INTO dim_payment_type (payment_type)
SELECT DISTINCT payment_type FROM stg_order_payments;

-- Fill the fact. One row per order item, as promised.
-- Payments are per order, so we take the first payment row for each order to avoid duplicating items.
INSERT INTO fact_sales (
    order_id, order_item_id, customer_key, product_key, seller_key,
    date_key, payment_type_key, price, freight_value, item_count, order_status
)
SELECT
    i.order_id, i.order_item_id, c.customer_key, p.product_key, s.seller_key,
    DATE_FORMAT(o.order_purchase_timestamp, '%Y%m%d'),
    pt.payment_type_key,
    i.price, i.freight_value, 1, o.order_status
FROM stg_order_items i
JOIN stg_orders o ON o.order_id = i.order_id
JOIN dim_customer c ON c.customer_id = o.customer_id
JOIN dim_product p ON p.product_id = i.product_id
JOIN dim_seller s ON s.seller_id = i.seller_id
LEFT JOIN (
    SELECT order_id, payment_type FROM (
        SELECT order_id, payment_type,
               ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY payment_sequential) AS rn
        FROM stg_order_payments
    ) ranked
    WHERE rn = 1
) first_pay ON first_pay.order_id = i.order_id
LEFT JOIN dim_payment_type pt ON pt.payment_type = first_pay.payment_type;

SELECT COUNT(*) AS fact_rows FROM fact_sales;
''', encoding="utf-8")

print("Wrote build_tables.sql")

Wrote build_tables.sql


In [9]:
# Task 9: Check that fact totals match the source totals
from pathlib import Path
import csv

Path("check_totals.sql").write_text(r'''-- Run both queries after build_tables.sql. The three numbers should match.

SELECT COUNT(*) AS fact_items,
       ROUND(SUM(price), 2) AS fact_price,
       ROUND(SUM(freight_value), 2) AS fact_freight
FROM fact_sales;

SELECT COUNT(*) AS source_items,
       ROUND(SUM(price), 2) AS source_price,
       ROUND(SUM(freight_value), 2) AS source_freight
FROM stg_order_items;
''', encoding="utf-8")

price_total = 0.0
freight_total = 0.0
item_count = 0
with open("../archive/olist_order_items_dataset.csv", newline="", encoding="utf-8-sig") as file:
    for row in csv.DictReader(file):
        price_total += float(row["price"])
        freight_total += float(row["freight_value"])
        item_count += 1

print("SQL written to check_totals.sql")
print("Source totals to match in MySQL:")
print("items:", item_count, "price:", round(price_total, 2), "freight:", round(freight_total, 2))

SQL written to check_totals.sql
Source totals to match in MySQL:
items: 112650 price: 13591643.7 freight: 2251909.54


In [10]:
# Task 10: Add fact_reviews and explain why it is a separate fact
from pathlib import Path

Path("fact_reviews.sql").write_text(r'''-- Reviews have their own grain: one row per review, not per order item.
-- Run build_tables.sql first so the staging tables and dim_product exist.

DROP TABLE IF EXISTS fact_reviews;

CREATE TABLE fact_reviews (
    review_key BIGINT AUTO_INCREMENT PRIMARY KEY,
    review_id CHAR(32),
    order_id CHAR(32),
    product_key INT,
    date_key INT,
    review_score TINYINT,
    has_comment BOOLEAN,
    answer_date_key INT,
    UNIQUE KEY uq_review (review_id)
);

CREATE TABLE IF NOT EXISTS stg_order_reviews (
    review_id CHAR(32), order_id CHAR(32), review_score TINYINT,
    review_comment_title VARCHAR(200), review_comment_message TEXT,
    review_creation_date DATETIME, review_answer_timestamp DATETIME
);

LOAD DATA LOCAL INFILE 'C:/path/archive/olist_order_reviews_dataset.csv'
INTO TABLE stg_order_reviews FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n' IGNORE 1 LINES;

INSERT INTO fact_reviews (
    review_id, order_id, product_key, date_key,
    review_score, has_comment, answer_date_key
)
SELECT
    r.review_id, r.order_id, p.product_key,
    DATE_FORMAT(r.review_creation_date, '%Y%m%d'),
    r.review_score,
    CASE WHEN r.review_comment_message IS NULL OR r.review_comment_message = '' THEN 0 ELSE 1 END,
    DATE_FORMAT(r.review_answer_timestamp, '%Y%m%d')
FROM stg_order_reviews r
LEFT JOIN stg_order_items i ON i.order_id = r.order_id AND i.order_item_id = 1
LEFT JOIN dim_product p ON p.product_id = i.product_id;
''', encoding="utf-8")

Path("fact_reviews.md").write_text(r'''# Why fact_reviews is a separate fact

I did not put reviews into `fact_sales` because the grain is different.

`fact_sales` has one row per **order item**. A review is written once for the **order**, usually after delivery. If I pasted the review score onto every item row, an order with four items would count that one review four times. That would make the average score wrong.

A separate `fact_reviews` table keeps one row per review, with its own measures: the score, whether the customer wrote a comment, and the dates the review was created and answered. It still joins to the same dimensions (date and product), so we can ask questions like "which product category gets the worst scores?" without distorting sales totals.
''', encoding="utf-8")

print("Wrote fact_reviews.sql and fact_reviews.md")

Wrote fact_reviews.sql and fact_reviews.md
